# 01 · Azure AI Document Intelligence — Copiloto de Oferta Pyme con Facturas

**Proyecto Integrador:** Copiloto de actualización de la **oferta crediticia pyme** de *Banco Andino del Pacífico* (entidad **ficticia**). El copiloto consulta la política de crédito pyme y lee las facturas electrónicas que presenta el cliente.


| Documento sintético | Archivo en `data/` | Modelo |
|---|---|---|
| Política de oferta pyme `POL-CRE-PYME-022` (3 páginas) | `Politica_Oferta_Pyme_Facturas_BAP.pdf` | `prebuilt-layout` |
| Factura ficticia (1 página) | `Factura_20999999975_E001-00000412.pdf` | `prebuilt-invoice` |

> Todos los documentos son sintéticos: emisores, clientes, RUC, montos y cuentas son ficticios y no tienen valor tributario.

## Paso 1 · Preparar la data sintética

Los documentos se generaron con IA y están en la carpeta `data/`:

- **Política `POL-CRE-PYME-022`:** documento principal. Describe cómo actualizar la oferta crediticia de un cliente pyme usando sus facturas recientes: requisitos, validación de facturas, cálculo de capacidad de pago, tablas de referencia, decisiones y un ejemplo.
- **Facturas electrónicas ficticias** de dos empresas pyme. Para la prueba de `prebuilt-invoice` se usa la factura `E001-00000412` de *Panificadora Santa Rosa del Norte E.I.R.L.*

In [16]:
# [adición del equipo] Rutas de los documentos sintéticos del proyecto
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
DOC_PRINCIPAL = str(ROOT / "data" / "Politica_Oferta_Pyme_Facturas_BAP.pdf")
DOC_FACTURA = str(ROOT / "data" / "Factura_20999999975_E001-00000412.pdf")
OUT_DIR = ROOT / "outputs" / "docintel"
OUT_DIR.mkdir(parents=True, exist_ok=True)

for ruta in (DOC_PRINCIPAL, DOC_FACTURA):
    print("OK   " if Path(ruta).exists() else "FALTA", Path(ruta).name)

OK    Politica_Oferta_Pyme_Facturas_BAP.pdf
OK    Factura_20999999975_E001-00000412.pdf


## Paso 2 · Configurar los clientes con la identidad ya validada

Se usa `DefaultAzureCredential` (sesión de `az login`). El endpoint se lee de `.env` (`DOCUMENT_INTELLIGENCE_ENDPOINT`). **No se usan API keys.**

In [17]:
# Librerias
import os
import json
from dotenv import load_dotenv
# (No se importa AzureKeyCredential: autenticación SOLO con identidad, sin API keys)
from azure.ai.documentintelligence import DocumentIntelligenceClient, DocumentIntelligenceAdministrationClient
from azure.core.exceptions import HttpResponseError
from azure.identity import DefaultAzureCredential


# Recargar variables del archivo .env
load_dotenv(override=True)


True

In [18]:
# Document Intelligence - Base Models
endpoint = os.getenv("DOCUMENT_INTELLIGENCE_ENDPOINT")

credential = DefaultAzureCredential()
client = DocumentIntelligenceClient(
    endpoint=endpoint,
    credential=credential,
)



In [19]:

# Resultado como markdown
def resultado_a_markdown(resultado, titulo="Documento procesado"):
    """
    Convierte el resultado de Azure AI Document Intelligence
    (por ejemplo, prebuilt-layout) a Markdown, incluyendo tablas.

    Args:
        resultado: Resultado de poller.result().
        titulo: Título del documento.

    Returns:
        str: Documento Markdown.
    """
    lineas_md = [
        f"# {titulo}",
        "",
        f"**Modelo utilizado:** `{resultado.model_id}`",
        f"**Número de páginas:** {len(resultado.pages or [])}",
        f"**Tablas detectadas:** {len(resultado.tables or [])}",
        "",
        "---",
        "",
    ]

    def limpiar(texto):
        """Evita que caracteres del contenido rompan la tabla Markdown."""
        if not texto:
            return ""
        return (
            str(texto)
            .replace("|", r"\|")
            .replace("\r", " ")
            .replace("\n", " ")
            .strip()
        )

    def tabla_a_markdown(tabla):
        """Convierte una tabla de Document Intelligence a Markdown."""
        filas = [
            [""] * tabla.column_count
            for _ in range(tabla.row_count)
        ]

        for celda in tabla.cells:
            filas[celda.row_index][celda.column_index] = limpiar(
                celda.content
            )

        # Elimina filas completamente vacías al principio y al final.
        while filas and not any(filas[0]):
            filas.pop(0)

        while filas and not any(filas[-1]):
            filas.pop()

        if not filas:
            return ""

        # Completa las filas para asegurar el mismo número de columnas.
        ancho = max(len(fila) for fila in filas)
        filas = [fila + [""] * (ancho - len(fila)) for fila in filas]

        salida = []
        salida.append("| " + " | ".join(filas[0]) + " |")
        salida.append("| " + " | ".join(["---"] * ancho) + " |")

        for fila in filas[1:]:
            salida.append("| " + " | ".join(fila) + " |")

        return "\n".join(salida)

    # Organiza las tablas por página de aparición, usando bounding_regions.
    tablas_por_pagina = {}

    for indice, tabla in enumerate(resultado.tables or [], start=1):
        regiones = getattr(tabla, "bounding_regions", None) or []

        paginas_tabla = {
            region.page_number for region in regiones
        } or {None}

        for numero_pagina in paginas_tabla:
            tablas_por_pagina.setdefault(numero_pagina, []).append(
                (indice, tabla)
            )

    for pagina in resultado.pages or []:
        numero_pagina = pagina.page_number

        lineas_md.extend([
            f"## Página {numero_pagina}",
            "",
        ])

        # Obtiene las tablas asociadas a esta página.
        tablas_pagina = tablas_por_pagina.get(numero_pagina, [])
        tablas_pagina_ids = {indice for indice, _ in tablas_pagina}

        # Texto detectado en la página.
        if pagina.lines:
            for linea in pagina.lines:
                texto = limpiar(linea.content)
                if texto:
                    lineas_md.extend([texto, ""])

        else:
            lineas_md.extend([
                "*No se detectó texto en esta página.*",
                "",
            ])

        # Muestra las tablas estructuradas detectadas en la página.
        if tablas_pagina:
            lineas_md.extend([
                f"### Tablas de la página {numero_pagina}",
                "",
            ])

            for indice, tabla in tablas_pagina:
                lineas_md.extend([
                    f"**Tabla {indice}**",
                    "",
                    tabla_a_markdown(tabla),
                    "",
                ])

    return "\n".join(lineas_md)



## Paso 3 · Listar los modelos disponibles

In [20]:
admin_client = DocumentIntelligenceAdministrationClient(
    endpoint=os.environ["DOCUMENT_INTELLIGENCE_ENDPOINT"],
    credential=credential
)

for modelo in admin_client.list_models():
    print("ID:", modelo.model_id,"| Descripción:", modelo.description)

ID: prebuilt-bankStatement.us | Descripción: Extract key information from US bank statement.
ID: prebuilt-check.us | Descripción: Extract key information from United States check.
ID: prebuilt-contract | Descripción: Extract key information from legal contracts.
ID: prebuilt-creditCard | Descripción: Extract key information from credit card.
ID: prebuilt-healthInsuranceCard.us | Descripción: Extract key information from US health insurance cards.
ID: prebuilt-idDocument | Descripción: Extract key information from passports and ID cards.
ID: prebuilt-invoice | Descripción: Extract key information from invoices.
ID: prebuilt-layout | Descripción: Extract text and layout information from documents.
ID: prebuilt-marriageCertificate.us | Descripción: Extract key information from US marriage certificate.
ID: prebuilt-mortgage.us.1003 | Descripción: Extract key information from US Uniform Residential Loan Application (Form 1003).
ID: prebuilt-mortgage.us.1004 | Descripción: Extract key inform

In [6]:
# [adición del equipo] Identificar los modelos prebuilt disponibles para el laboratorio
prebuilt = [m.model_id for m in admin_client.list_models() if m.model_id.startswith("prebuilt-")]
print(f"Modelos prebuilt disponibles: {len(prebuilt)}")
print("¿prebuilt-layout disponible? ", "prebuilt-layout" in prebuilt)
print("¿prebuilt-invoice disponible?", "prebuilt-invoice" in prebuilt)

Modelos prebuilt disponibles: 66
¿prebuilt-layout disponible?  True
¿prebuilt-invoice disponible? True


## Paso 4 · Probar `prebuilt-layout` sobre el documento principal

Procedimiento de clase; solo cambia la ruta del documento y el título.

In [21]:
# Documento sintético principal del proyecto: política de oferta pyme
ruta_documento = DOC_PRINCIPAL

# Ejemplo usando prebuilt-read
with open(ruta_documento, "rb") as archivo:
    poller = client.begin_analyze_document(
        model_id="prebuilt-layout",
        body=archivo,
        content_type="application/octet-stream",
    )

resultado = poller.result()

print("Análisis completado")
print("Modelo utilizado:", resultado.model_id)
print("Número de páginas:", len(resultado.pages))

# Se muestra el resultado
markdown = resultado_a_markdown(resultado,titulo="POL-CRE-PYME-022 - Política de oferta pyme con facturas")
print(markdown)

Análisis completado
Modelo utilizado: prebuilt-layout
Número de páginas: 3
# POL-CRE-PYME-022 - Política de oferta pyme con facturas

**Modelo utilizado:** `prebuilt-layout`
**Número de páginas:** 3
**Tablas detectadas:** 7

---

## Página 1

POL-CRE-PYME-022 · Versión 1.0 · Uso interno

BANCO ANDINO DEL PACÍFICO

Política de actualización de la oferta crediticia pyme con

facturas recientes

Código / vigencia

POL-CRE-PYME-022 · vigente desde 01/10/2026

Área responsable

Gerencia de Riesgos Pyme

Aplica a

Préstamo Capital de Trabajo Pyme y Línea Revolvente Pyme

1. Objetivo

La oferta preaprobada de cada cliente pyme se calcula con la información disponible en el banco

(último estado financiero, ingresos declarados o comportamiento de pagos), que puede estar

desactualizada. Esta política define cómo usar las facturas electrónicas emitidas por el cliente

para recalcular su capacidad de pago y, si corresponde, proponer una mejora de la oferta. La

propuesta nunca se comunica al cli

### 4.1 Revisión de páginas, secciones y tablas 


In [ ]:
# Resumen: páginas, texto, secciones y tablas
print("Páginas:", len(resultado.pages))
print("Párrafos:", len(resultado.paragraphs or []))
print("Secciones:", len(resultado.sections or []))
print("Tablas:", len(resultado.tables or []))
print("Caracteres de texto:", len(resultado.content))

print("\nEncabezados de sección detectados:")
for p in resultado.paragraphs:
    if p.role in ("title", "sectionHeading"):
        pagina = p.bounding_regions[0].page_number if p.bounding_regions else "?"
        print(f"  [pág. {pagina}] {p.content}")

print("\nTablas detectadas:")
for i, t in enumerate(resultado.tables, start=1):
    pagina = t.bounding_regions[0].page_number if t.bounding_regions else "?"
    encabezado = [c.content for c in t.cells if c.row_index == 0]
    print(f"  Tabla {i}: página {pagina}, {t.row_count} filas x {t.column_count} columnas | {encabezado}")

# Evidencia
layout_resultado = resultado
(OUT_DIR / "politica_layout.md").write_text(markdown, encoding="utf-8")
(OUT_DIR / "layout_result_raw.json").write_text(json.dumps(resultado.as_dict(), ensure_ascii=False, indent=2), encoding="utf-8")
print("\nGuardados: outputs/docintel/politica_layout.md y outputs/docintel/layout_result_raw.json")

Páginas: 3
Párrafos: 138
Secciones: 8
Tablas: 7
Caracteres de texto: 6304

Encabezados de sección detectados:
  [pág. 1] Política de actualización de la oferta crediticia pyme con facturas recientes
  [pág. 1] 1. Objetivo
  [pág. 1] 2. Requisitos para la reevaluación
  [pág. 1] 3. Validación de cada factura
  [pág. 1] 4. Cálculo de la capacidad de pago
  [pág. 2] 5. Decisión sobre la oferta
  [pág. 2] 6. Señales de alerta y límites del agente
  [pág. 3] 7. Ejemplo resuelto (fecha de evaluación: 05/11/2026)

Tablas detectadas:
  Tabla 1: página 1, 3 filas x 2 columnas | ['Código / vigencia', 'POL-CRE-PYME-022 · vigente desde 01/10/2026']
  Tabla 2: página 1, 5 filas x 2 columnas | ['Requisito', 'Condición']
  Tabla 3: página 1, 8 filas x 3 columnas | ['Regla', 'Campo extraído', 'Criterio']
  Tabla 4: página 2, 6 filas x 2 columnas | ['Sector (CIIU agrupado)', 'Margen']
  Tabla 5: página 2, 6 filas x 2 columnas | ['Plazo', 'Factor']
  Tabla 6: página 2, 6 filas x 2 columnas | ['Resultado

**Observaciones de `prebuilt-layout`:**
- Se detectaron las **3 páginas**, **138 párrafos**, **8 secciones** y **7 tablas** de la política.
- Se reconocieron como encabezados de sección el título y los 7 apartados numerados (1. Objetivo … 7. Ejemplo resuelto), cada uno con su página.
- El encabezado y el pie que se repiten en cada página ("BANCO ANDINO DEL PACÍFICO · POL-CRE-PYME-022…", "Página N") se identificaron como `pageHeader`, `pageFooter` y `pageNumber`, así que es fácil excluirlos del contenido.
- Las tablas conservan su estructura fila-columna, incluidas las celdas con textos largos (criterios de validación, acciones del agente).
- En el Markdown por líneas de la función de clase, el texto de las tablas aparece primero como líneas sueltas y después como tabla estructurada. Por eso, para el JSON del Paso 6 se usan las **secciones**, que separan párrafos y tablas sin duplicar contenido.

## Paso 5 · Probar `prebuilt-invoice` sobre la factura sintética

Procedimiento de clase; solo cambia la ruta del documento.

In [22]:
# Factura sintética del proyecto
ruta_documento = DOC_FACTURA

# Ejemplo usando prebuilt-invoice
with open(ruta_documento, "rb") as archivo:
    poller = client.begin_analyze_document(
        model_id="prebuilt-invoice",
        body=archivo,
        content_type="application/octet-stream",
    )

invoices = poller.result()

json_pretty = json.dumps(invoices.as_dict(), indent=6)
print(json_pretty)

{
      "apiVersion": "2024-11-30",
      "modelId": "prebuilt-invoice",
      "stringIndexType": "textElements",
      "content": "Panificadora Santa Rosa del Norte\nRUC: 20999999975\nE.I.R.L.\nAv. T\u00fapac Amaru 2140, Comas, Lima, Per\u00fa\nFACTURA ELECTR\u00d3NICA\nTel\u00e9fono: +51 1 555 0187\nCorreo: ventas@santarosa-ficticio.pe\nE001-00000412\nSe\u00f1or(es):\nHotel Mirador del R\u00edmac S.A.\nFecha de emisi\u00f3n:\n05/08/2026\nRUC:\n20999999959\nFecha de vencimiento:\n04/09/2026\nDirecci\u00f3n:\nAv. Arequipa 4120, Miraflores, Lima\nOrden de compra:\nOC-HTL-0931\nForma de pago:\nCr\u00e9dito a 30 d\u00edas\nMoneda:\nSoles (PEN)\nC\u00f3digo\nDescripci\u00f3n\nUnidad\nCant.\nP. unitario\nDscto.\nImporte\nPAN-01\nPan franc\u00e9s (ciento)\nCiento\n120\nS/ 18.00\n\u2014\nS/ 2,160.00\nPAN-07\nCroissant de mantequilla\nUnidad\n900\nS/ 2.20\n-\nS/ 1,980.00\nPST-03\nTorta de chocolate 24\nporciones\nUnidad\n14\nS/ 95.00\n-\nS/ 1,330.00\nOp. gravada\nRIQ\nS/ 5,470.00\nIGV (18%)\nS

In [10]:

if invoices.documents:
    for idx, invoice in enumerate(invoices.documents):
        print(f"--------Analyzing invoice #{idx + 1}--------")
        vendor_name = invoice.fields.get("VendorName")
        if vendor_name:
            print(
                f"Vendor Name: {vendor_name.get('content')} has confidence: {vendor_name.get('confidence')}"
            )
        vendor_address = invoice.fields.get("VendorAddress")
        if vendor_address:
            print(
                f"Vendor Address: {vendor_address.get('content')} has confidence: {vendor_address.get('confidence')}"
            )
        vendor_address_recipient = invoice.fields.get("VendorAddressRecipient")
        if vendor_address_recipient:
            print(
                f"Vendor Address Recipient: {vendor_address_recipient.get('content')} has confidence: {vendor_address_recipient.get('confidence')}"
            )
        customer_name = invoice.fields.get("CustomerName")
        if customer_name:
            print(
                f"Customer Name: {customer_name.get('content')} has confidence: {customer_name.get('confidence')}"
            )
        customer_id = invoice.fields.get("CustomerId")
        if customer_id:
            print(
                f"Customer Id: {customer_id.get('content')} has confidence: {customer_id.get('confidence')}"
            )
        customer_address = invoice.fields.get("CustomerAddress")
        if customer_address:
            print(
                f"Customer Address: {customer_address.get('content')} has confidence: {customer_address.get('confidence')}"
            )
        customer_address_recipient = invoice.fields.get("CustomerAddressRecipient")
        if customer_address_recipient:
            print(
                f"Customer Address Recipient: {customer_address_recipient.get('content')} has confidence: {customer_address_recipient.get('confidence')}"
            )
        invoice_id = invoice.fields.get("InvoiceId")
        if invoice_id:
            print(
                f"Invoice Id: {invoice_id.get('content')} has confidence: {invoice_id.get('confidence')}"
            )
        invoice_date = invoice.fields.get("InvoiceDate")
        if invoice_date:
            print(
                f"Invoice Date: {invoice_date.get('content')} has confidence: {invoice_date.get('confidence')}"
            )
        invoice_total = invoice.fields.get("InvoiceTotal")
        if invoice_total:
            print(
                f"Invoice Total: {invoice_total.get('content')} has confidence: {invoice_total.get('confidence')}"
            )
        due_date = invoice.fields.get("DueDate")
        if due_date:
            print(
                f"Due Date: {due_date.get('content')} has confidence: {due_date.get('confidence')}"
            )
        purchase_order = invoice.fields.get("PurchaseOrder")
        if purchase_order:
            print(
                f"Purchase Order: {purchase_order.get('content')} has confidence: {purchase_order.get('confidence')}"
            )
        billing_address = invoice.fields.get("BillingAddress")
        if billing_address:
            print(
                f"Billing Address: {billing_address.get('content')} has confidence: {billing_address.get('confidence')}"
            )
        billing_address_recipient = invoice.fields.get("BillingAddressRecipient")
        if billing_address_recipient:
            print(
                f"Billing Address Recipient: {billing_address_recipient.get('content')} has confidence: {billing_address_recipient.get('confidence')}"
            )
        shipping_address = invoice.fields.get("ShippingAddress")
        if shipping_address:
            print(
                f"Shipping Address: {shipping_address.get('content')} has confidence: {shipping_address.get('confidence')}"
            )
        shipping_address_recipient = invoice.fields.get("ShippingAddressRecipient")
        if shipping_address_recipient:
            print(
                f"Shipping Address Recipient: {shipping_address_recipient.get('content')} has confidence: {shipping_address_recipient.get('confidence')}"
            )
        print("Invoice items:")
        items = (invoice.fields.get("Items") or {}).get("valueArray") or []
        for idx, item in enumerate(items):
            print(f"...Item #{idx + 1}")
            item_description = (item.get("valueObject") or {}).get("Description")
            if item_description:
                print(
                    f"......Description: {item_description.get('content')} has confidence: {item_description.get('confidence')}"
                )
            item_quantity = (item.get("valueObject") or {}).get("Quantity")
            if item_quantity:
                print(
                    f"......Quantity: {item_quantity.get('content')} has confidence: {item_quantity.get('confidence')}"
                )
            unit = (item.get("valueObject") or {}).get("Unit")
            if unit:
                print(
                    f"......Unit: {unit.get('content')} has confidence: {unit.get('confidence')}"
                )
            unit_price = (item.get("valueObject") or {}).get("UnitPrice")
            if unit_price:
                unit_price_code = (unit_price.get("valueCurrency") or {}).get("currencyCode") or ""
                print(
                    f"......Unit Price: {unit_price.get('content')}{unit_price_code} has confidence: {unit_price.get('confidence')}"
                )
            product_code = (item.get("valueObject") or {}).get("ProductCode")
            if product_code:
                print(
                    f"......Product Code: {product_code.get('content')} has confidence: {product_code.get('confidence')}"
                )
            item_date = (item.get("valueObject") or {}).get("Date")
            if item_date:
                print(
                    f"......Date: {item_date.get('content')} has confidence: {item_date.get('confidence')}"
                )
            tax = (item.get("valueObject") or {}).get("Tax")
            if tax:
                print(
                    f"......Tax: {tax.get('content')} has confidence: {tax.get('confidence')}"
                )
            amount = (item.get("valueObject") or {}).get("Amount")
            if amount:
                print(
                    f"......Amount: {amount.get('content')} has confidence: {amount.get('confidence')}"
                )
        subtotal = invoice.fields.get("SubTotal")
        if subtotal:
            print(
                f"Subtotal: {subtotal.get('content')} has confidence: {subtotal.get('confidence')}"
            )
        total_tax = invoice.fields.get("TotalTax")
        if total_tax:
            print(
                f"Total Tax: {total_tax.get('content')} has confidence: {total_tax.get('confidence')}"
            )
        previous_unpaid_balance = invoice.fields.get("PreviousUnpaidBalance")
        if previous_unpaid_balance:
            print(
                f"Previous Unpaid Balance: {previous_unpaid_balance.get('content')} has confidence: {previous_unpaid_balance.get('confidence')}"
            )
        amount_due = invoice.fields.get("AmountDue")
        if amount_due:
            print(
                f"Amount Due: {amount_due.get('content')} has confidence: {amount_due.get('confidence')}"
            )
        service_start_date = invoice.fields.get("ServiceStartDate")
        if service_start_date:
            print(
                f"Service Start Date: {service_start_date.get('content')} has confidence: {service_start_date.get('confidence')}"
            )
        service_end_date = invoice.fields.get("ServiceEndDate")
        if service_end_date:
            print(
                f"Service End Date: {service_end_date.get('content')} has confidence: {service_end_date.get('confidence')}"
            )
        service_address = invoice.fields.get("ServiceAddress")
        if service_address:
            print(
                f"Service Address: {service_address.get('content')} has confidence: {service_address.get('confidence')}"
            )
        service_address_recipient = invoice.fields.get("ServiceAddressRecipient")
        if service_address_recipient:
            print(
                f"Service Address Recipient: {service_address_recipient.get('content')} has confidence: {service_address_recipient.get('confidence')}"
            )
        remittance_address = invoice.fields.get("RemittanceAddress")
        if remittance_address:
            print(
                f"Remittance Address: {remittance_address.get('content')} has confidence: {remittance_address.get('confidence')}"
            )
        remittance_address_recipient = invoice.fields.get(
            "RemittanceAddressRecipient"
        )
        if remittance_address_recipient:
            print(
                f"Remittance Address Recipient: {remittance_address_recipient.get('content')} has confidence: {remittance_address_recipient.get('confidence')}"
            )

        print("----------------------------------------")

--------Analyzing invoice #1--------
Vendor Name: Panificadora Santa Rosa del Norte
E.I.R.L. has confidence: 0.72
Vendor Address: Av. Túpac Amaru 2140, Comas, Lima, Perú has confidence: 0.893
Vendor Address Recipient: Panificadora Santa Rosa del Norte
E.I.R.L. has confidence: 0.72
Customer Name: Hotel Mirador del Rímac S.A. has confidence: 0.889
Customer Address: Av. Arequipa 4120, Miraflores, Lima has confidence: 0.919
Customer Address Recipient: Hotel Mirador del Rímac S.A. has confidence: 0.889
Invoice Id: E001-00000412 has confidence: 0.959
Invoice Date: 05/08/2026 has confidence: 0.938
Invoice Total: S/ 6,454.60 has confidence: 0.926
Due Date: 04/09/2026 has confidence: 0.947
Purchase Order: OC-HTL-0931 has confidence: 0.936
Invoice items:
...Item #1
......Description: Pan francés (ciento) has confidence: 0.938
......Quantity: 120 has confidence: 0.94
......Unit: Ciento has confidence: 0.934
......Unit Price: S/ 18.00PEN has confidence: 0.943
......Product Code: PAN-01 has confide

In [15]:
# Resumen: páginas, texto detectado y tablas de la factura
print("Modelo:", invoices.model_id)
print("Páginas:", len(invoices.pages))
print("Tipo de documento:", invoices.documents[0].doc_type, "| confianza:", invoices.documents[0].confidence)
print("Campos identificados:", len(invoices.documents[0].fields), "->", ", ".join(invoices.documents[0].fields))
print("Tablas:", len(invoices.tables or []))

print("\n--- Texto detectado ---")
print(invoices.content)
print("\n--- Tablas (Markdown) ---")
print(resultado_a_markdown(invoices, titulo="Factura E001-00000412").split("### Tablas", 1)[-1])

(OUT_DIR / "invoice_result_raw.json").write_text(json.dumps(invoices.as_dict(), ensure_ascii=False, indent=2), encoding="utf-8")
print("Guardado: outputs/docintel/invoice_result_raw.json")

Modelo: prebuilt-invoice
Páginas: 1
Tipo de documento: invoice | confianza: 1.0
Campos identificados: 18 -> CustomerAddress, CustomerAddressRecipient, CustomerName, CustomerTaxId, DueDate, InvoiceDate, InvoiceId, InvoiceTotal, Items, PaymentTerm, PurchaseOrder, SubTotal, TaxDetails, TotalTax, VendorAddress, VendorAddressRecipient, VendorName, VendorTaxId
Tablas: 4

--- Texto detectado ---
Panificadora Santa Rosa del Norte
RUC: 20999999975
E.I.R.L.
Av. Túpac Amaru 2140, Comas, Lima, Perú
FACTURA ELECTRÓNICA
Teléfono: +51 1 555 0187
Correo: ventas@santarosa-ficticio.pe
E001-00000412
Señor(es):
Hotel Mirador del Rímac S.A.
Fecha de emisión:
05/08/2026
RUC:
20999999959
Fecha de vencimiento:
04/09/2026
Dirección:
Av. Arequipa 4120, Miraflores, Lima
Orden de compra:
OC-HTL-0931
Forma de pago:
Crédito a 30 días
Moneda:
Soles (PEN)
Código
Descripción
Unidad
Cant.
P. unitario
Dscto.
Importe
PAN-01
Pan francés (ciento)
Ciento
120
S/ 18.00
—
S/ 2,160.00
PAN-07
Croissant de mantequilla
Unidad
900


## Paso 6 · Guardar el resultado en un archivo JSON

Se elige el resultado de **`prebuilt-layout` de la política**, porque es la información que consultará el Copiloto. Se aprovechan las **secciones** detectadas por el servicio: cada sección se convierte en un documento indexable en Azure AI Search, con su texto y sus tablas.

Campos: `id`, `title`, `content` (mínimos) + metadatos `category`, `documentType`, `sourceFile`, `pageNumber`, `source`, `createdAt`.

In [12]:
# [adición del equipo] Transformación a JSON para Azure AI Search (un documento por sección)
import re
from datetime import date

def slugify(texto):
    texto = texto.lower()
    for a, b in zip("áéíóúñü", "aeiounu"):
        texto = texto.replace(a, b)
    return re.sub(r"[^a-z0-9]+", "-", texto).strip("-")[:60]

def tabla_md(tabla):
    filas = [[""] * tabla.column_count for _ in range(tabla.row_count)]
    for celda in tabla.cells:
        filas[celda.row_index][celda.column_index] = (celda.content or "").replace("|", "/").replace("\n", " ")
    salida = ["| " + " | ".join(filas[0]) + " |", "| " + " | ".join(["---"] * tabla.column_count) + " |"]
    salida += ["| " + " | ".join(f) + " |" for f in filas[1:]]
    return "\n".join(salida)

ROLES_EXCLUIDOS = {"pageHeader", "pageFooter", "pageNumber"}

documentos_busqueda = []
for seccion in layout_resultado.sections:
    encabezado, pagina, partes = None, None, []
    for elemento in seccion.elements:
        _, tipo, idx = elemento.split("/")
        if tipo == "paragraphs":
            p = layout_resultado.paragraphs[int(idx)]
            if p.role in ROLES_EXCLUIDOS:
                continue
            if p.role in ("sectionHeading", "title") and encabezado is None:
                encabezado = p.content
                pagina = p.bounding_regions[0].page_number if p.bounding_regions else None
                continue
            partes.append(p.content)
        elif tipo == "tables":
            partes.append(tabla_md(layout_resultado.tables[int(idx)]))
    if not encabezado or not partes:
        continue
    documentos_busqueda.append({
        "id": f"pol-cre-pyme-022-{len(documentos_busqueda) + 1:02d}-{slugify(encabezado)}",
        "title": encabezado,
        "content": "\n\n".join(partes),
        "category": "credito-pyme",
        "documentType": "politica-credito",
        "sourceFile": Path(DOC_PRINCIPAL).name,
        "pageNumber": pagina,
        "source": "sintetico",
        "createdAt": date.today().isoformat(),
    })

ruta_json = OUT_DIR / "pol_cre_pyme_022_search_index.json"
ruta_json.write_text(json.dumps(documentos_busqueda, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Guardado {ruta_json.relative_to(ROOT)} con {len(documentos_busqueda)} documentos:\n")
for d in documentos_busqueda:
    print(f"  pág. {d['pageNumber']} | {d['title']} ({len(d['content'])} caracteres)")

Guardado outputs/docintel/pol_cre_pyme_022_search_index.json con 8 documentos:

  pág. 1 | Política de actualización de la oferta crediticia pyme con facturas recientes (201 caracteres)
  pág. 1 | 1. Objetivo (443 caracteres)
  pág. 1 | 2. Requisitos para la reevaluación (494 caracteres)
  pág. 1 | 3. Validación de cada factura (1245 caracteres)
  pág. 1 | 4. Cálculo de la capacidad de pago (1169 caracteres)
  pág. 2 | 5. Decisión sobre la oferta (682 caracteres)
  pág. 2 | 6. Señales de alerta y límites del agente (479 caracteres)
  pág. 3 | 7. Ejemplo resuelto (fecha de evaluación: 05/11/2026) (1091 caracteres)


In [ ]:
# Ejemplo de un documento del JSON (sección con tabla)
ejemplo = next((d for d in documentos_busqueda if "| ---" in d["content"] and "Regla" in d["content"]), documentos_busqueda[0])
print(json.dumps(ejemplo, ensure_ascii=False, indent=2))

{
  "id": "pol-cre-pyme-022-04-3-validacion-de-cada-factura",
  "title": "3. Validación de cada factura",
  "content": "Cada factura se procesa con Document Intelligence (modelo prebuilt-invoice) y solo se considera válida si cumple todas las reglas siguientes. Las facturas que no las cumplan se excluyen del cálculo y se informa el motivo.\n\n| Regla | Campo extraído | Criterio |\n| --- | --- | --- |\n| V1 Emisor | VendorTaxId | Debe ser igual al RUC del cliente evaluado. Facturas recibidas (compras) no sustentan ingresos. |\n| V2 Antigüedad | InvoiceDate | Dentro de los últimos 6 meses respecto de la fecha de evaluación y nunca posterior a ella. |\n| V3 Aritmética | Items, SubTotal, TotalTax, InvoiceTotal | La suma de ítems debe igualar el SubTotal y SubTotal + IGV debe igualar el total (tolerancia S/ 1.00). |\n| V4 Duplicados | InvoiceId | Un mismo número de serie y correlativo cuenta una sola vez. |\n| V5 Partes relacionadas | CustomerTaxId | No cuentan las ventas a empresas del mis